# Lab 12: LSTM Model for Time-Series Forecasting

**Student Name:** DANIYAL BASHARAT  
**Registration Number:** 22JZELE0467  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Lab Overview
This notebook develops an LSTM-based deep learning model for time-series forecasting. The code imports forecasting utilities, defines an LSTM architecture, configures callbacks and checkpoints, loads train/validation/test datasets, trains the model, and evaluates forecasting performance using regression metrics.

## Learning Objectives
- Set the working directory and import Scikit-learn, TensorFlow/Keras, and custom time-series utilities.
- Define an LSTM neural network architecture for sequential forecasting data.
- Configure optimizer, model checkpoints, and training monitor callbacks.
- Load training, validation, and testing CSV files for time-series prediction.
- Evaluate LSTM forecasting output using MAE, MSE, RMSE, R2 score, and explained variance.

## Notebook Format
The original code cells and existing outputs have been kept unchanged. Additional headings and comments are added only as Markdown to make the notebook more professional and easier to follow.


## Section 1: Working Directory and Library Imports
This section sets the Lab 12 working directory and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utility functions.


In [2]:
import os
os.chdir('Z:\\University\\8th Semester\\ML Lab\\Lab 12')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## Section 2: Model Parameters and LSTM Architecture
The following cells define time steps, feature count, and the LSTM model architecture used for sequential forecasting.


In [6]:
model1 = create_lstm()
model1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 8)          │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 20)             │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,301 (12.89 KB)

 Trainable params: 3,301 (12.89 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [8]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'Z:\University\8th Semester\ML Lab\Lab 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## Section 3: Checkpoints, Callbacks, and Training Configuration
This section prepares checkpoint paths, output directories, training monitor callbacks, optimizer settings, and model compilation.


In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [11]:
path_dataset = r'Z:\University\8th Semester\ML Lab\Lab 12'

path_tr = os.path.join(path_dataset, 'AEP_train.csv')
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
path_te = os.path.join(path_dataset, 'AEP_test.csv')
path_hourly = os.path.join(path_dataset, 'AEP_hourly.csv')

for path in [path_tr, path_v, path_te, path_hourly]:
    print(path, "=>", os.path.exists(path))

df_tr = pd.read_csv(path_tr)
train_set = df_tr.values

df_v = pd.read_csv(path_v)
validation_set = df_v.values

df_te = pd.read_csv(path_te)
test_set = df_te.values

df_hourly = pd.read_csv(path_hourly)

train_set.shape, validation_set.shape, test_set.shape, df_hourly.shape

Z:\University\8th Semester\ML Lab\Lab 12\AEP_train.csv => True
Z:\University\8th Semester\ML Lab\Lab 12\AEP_validation.csv => True
Z:\University\8th Semester\ML Lab\Lab 12\AEP_test.csv => True
Z:\University\8th Semester\ML Lab\Lab 12\AEP_hourly.csv => True


((84907, 21), (24259, 21), (12130, 21), (121273, 2))

In [12]:
time_steps=24
num_features=21

In [13]:
start = time.time()

train_X, train_y = univariate_multi_step(
    train_set,
    time_steps,
    target_col=0,
    target_len=1
)

validation_X, validation_y = univariate_multi_step(
    validation_set,
    time_steps,
    target_col=0,
    target_len=1
)

test_X, test_y = univariate_multi_step(
    test_set,
    time_steps,
    target_col=0,
    target_len=1
)

print('Time Consumed', time.time() - start, "sec")

Time Consumed 0.37630462646484375 sec


## Section 4: Dataset Loading and Forecast Preparation
The following cells load train, validation, and test CSV files, then prepare the data for LSTM forecasting input/output format.


In [ ]:
epochs = 60
verbose = 1
batch_size = 32

History = model.fit(
    train_X,
    train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=verbose
)

Epoch 1/60
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0714 - mae: 0.0714 - mape: 73.9250
Epoch 1: val_loss improved from None to 0.02361, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0001-loss0.02.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0001-loss0.02.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.0450 - mae: 0.0450 - mape: 584.9526 - val_loss: 0.0236 - val_mae: 0.0236 - val_mape: 10.1161
Epoch 2/60
2647/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0216 - mae: 0.0216 - mape: 224.7906
Epoch 2: val_loss improved from 0.02361 to 0.01882, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0002-loss0.02.h5



Epoch 2: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0002-loss0.02.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.0195 - mae: 0.0195 - mape: 341.9720 - val_loss: 0.0188 - val_mae: 0.0188 - val_mape: 7.9462
Epoch 3/60
1070/2653 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0157 - mae: 0.0157 - mape: 5.1236

In [ ]:
import os
import pickle

path_dataset = r'Z:\University\8th Semester\ML Lab\Lab 12'
path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')

print(os.path.exists(path_scaler))

with open(path_scaler, 'rb') as f:
    scaler = pickle.load(f)

True


c:\Users\engin\.conda\envs\ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
model_path = r'Z:\University\8th Semester\ML Lab\Lab 12\E1-cp-0032-loss0.01.h5'

model = load_model(model_path, compile=False)

y_pred_scaled = model.predict(test_X)

print("y_pred_scaled.shape =", y_pred_scaled.shape)
print("test_y.shape =", test_y.shape)

def inverse_target(scaled_values, scaler, target_col=0, num_features=21):
    scaled_values = scaled_values.reshape(-1)

    dummy = np.zeros((scaled_values.shape[0], num_features))
    dummy[:, target_col] = scaled_values

    unscaled = scaler.inverse_transform(dummy)

    return unscaled[:, target_col].reshape(-1, 1)

y_pred = inverse_target(
    y_pred_scaled,
    scaler,
    target_col=0,
    num_features=21
)

y_test_unscaled = inverse_target(
    test_y,
    scaler,
    target_col=0,
    num_features=21
)

# Mean Absolute Error (MAE)
MAE = np.mean(np.abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(np.abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.mean(np.square(y_pred - y_test_unscaled))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(MSE)
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape = ', y_test_unscaled.shape)
print('y_pred.shape = ', y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
y_pred_scaled.shape = (12105, 1)
test_y.shape = (12105, 1)
Mean Absolute Error (MAE): 101.25
Median Absolute Error (MedAE): 79.74
Mean Squared Error (MSE): 18440.83
Root Mean Squared Error (RMSE): 135.8
Mean Absolute Percentage Error (MAPE): 0.69 %
Median Absolute Percentage Error (MDAPE): 0.55 %


y_test_unscaled.shape =  (12105, 1)
y_pred.shape =  (12105, 1)


In [ ]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'Z:\University\8th Semester\ML Lab\Lab 12\E1-cp-0032-loss0.01.h5'
start_epoch= 34

## Section 5: Prediction and Model Evaluation
The final cells generate predictions and calculate regression metrics to evaluate the forecasting model performance.


In [ ]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_load = r'Z:\University\8th Semester\ML Lab\Lab 12\E1-cp-0032-loss0.01.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_load))
    model = load_model(model_load, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )


    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading Z:\University\8th Semester\ML Lab\Lab 12\E1-cp-0032-loss0.01.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [ ]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
2644/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0059 - mae: 0.0059 - mape: 6.2753
Epoch 1: val_loss improved from None to 0.00589, saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0001-loss0.01.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0001-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 0.0059 - mae: 0.0059 - mape: 23.8518 - val_loss: 0.0059 - val_mae: 0.0059 - val_mape: 2.4671
Epoch 2/10
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0058 - mae: 0.0058 - mape: 8.7523
Epoch 2: val_loss did not improve from 0.00589
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - loss: 0.0058 - mae: 0.0058 - mape: 10.6644 - val_loss: 0.0060 - val_mae: 0.0060 - val_mape: 2.4916
Epoch 3/10
2648/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 8.6652
Epoch 3: val_loss did not improve from 0.00589
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 8.5350 - val_loss: 0.0059 - val_mae: 0.0059 - val_mape: 2.5555
Epoch 4/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 3.4054
Epoch 4: val_loss did not improve from 0.00589
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 2


Epoch 7: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0007-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 9.8000 - val_loss: 0.0059 - val_mae: 0.0059 - val_mape: 2.4835
Epoch 8/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 2.2494
Epoch 8: val_loss did not improve from 0.00588
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 55.3073 - val_loss: 0.0059 - val_mae: 0.0059 - val_mape: 2.4705
Epoch 9/10
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0058 - mae: 0.0058 - mape: 15.9539
Epoch 9: val_loss improved from 0.00588 to 0.00579, saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0009-loss0.01.h5



Epoch 9: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0009-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 25s 9ms/step - loss: 0.0058 - mae: 0.0058 - mape: 41.6505 - val_loss: 0.0058 - val_mae: 0.0058 - val_mape: 2.4306
Epoch 10/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0058 - mae: 0.0058 - mape: 2.3671
Epoch 10: val_loss did not improve from 0.00579
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - loss: 0.0058 - mae: 0.0058 - mape: 38.7539 - val_loss: 0.0059 - val_mae: 0.0059 - val_mape: 2.5035


In [ ]:
model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0009-loss0.01.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Mean Absolute Error (MAE): 97.2
Median Absolute Error (MedAE): 76.04
Mean Squared Error (MSE): 17044.96
Root Mean Squared Error (RMSE): 130.56
Mean Absolute Percentage Error (MAPE): 0.66 %
Median Absolute Percentage Error (MDAPE): 0.53 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Final Conclusion
In this lab, an LSTM neural network was implemented for time-series forecasting. The notebook demonstrates sequence-model architecture, callback-based training, data preparation, and regression-based model evaluation.

## Submitted By
**Student Name:** DANIYAL BASHARAT  
**Registration Number:** 22JZELE0467  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus
